# Whitespace Locations - Silver Layer

Generate expansion candidate locations from top 25% H3 cells by total POI count.
These locations represent areas with high commercial activity that could be good candidates for new store openings.

**Note:** Whitespace locations are filtered to MA only, while H3 features include all training states (MA, CT, NJ, MD).

**Inputs:**
- `{catalog}.{silver_schema}.h3_features_clean` - Clean H3 features with derived columns (filtered to MA)
- `{catalog}.{bronze_schema}.current_stores_ne` - Current store locations for distance calculation (all states)

**Output:**
- `{catalog}.{silver_schema}.whitespace_locations` - Expansion candidate locations (MA only)

**Output Schema:**
- location_id (starting at 999001)
- store_type = 'Expansion Candidate'
- latitude, longitude (H3 centroid)
- address, city, zip_code = 'NA'
- state = from H3 feature data
- country_code = 'US'
- geo_accuracy = 'H3_CENTROID'
- distance_to_nearest_current_store (Haversine km)
- h3_cell_id, total_poi_count, population, urbanity

## Parameters

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit, row_number, udf, broadcast
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
import math

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("state_filter", "MA")
dbutils.widgets.text("location_id_start", "999001")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
state_filter = dbutils.widgets.get("state_filter")
location_id_start = int(dbutils.widgets.get("location_id_start"))

assert catalog and bronze_schema and silver_schema, "Missing required parameters"

# Table names
h3_features_table = f"{catalog}.{silver_schema}.h3_features_clean"
current_stores_table = f"{catalog}.{bronze_schema}.current_stores_ne"
output_table = f"{catalog}.{silver_schema}.whitespace_locations"

print(f"Catalog: {catalog}")
print(f"H3 features: {h3_features_table}")
print(f"Current stores: {current_stores_table}")
print(f"Output table: {output_table}")
print(f"State filter: {state_filter}")
print(f"Location ID start: {location_id_start}")

## Load H3 Features and Filter to Top 25%

In [ ]:
# Load clean H3 features and filter to state_filter (MA) for whitespace locations
# Note: h3_features_clean contains all training states (MA, CT, NJ, MD), but whitespace locations are MA only
h3_features_all = spark.table(h3_features_table)

# Filter to target state for whitespace locations
h3_features = h3_features_all.filter(col("state_abbr") == state_filter)

total_cells_all = h3_features_all.count()
total_cells = h3_features.count()
print(f"Total H3 cells in h3_features_clean: {total_cells_all:,}")
print(f"Filtered to {state_filter}: {total_cells:,} H3 cells")

# Calculate 75th percentile threshold for total_poi_count (within filtered state)
poi_percentile = h3_features.filter(col("total_poi_count") > 0).selectExpr(
    "percentile_approx(total_poi_count, 0.75) as p75"
).collect()[0]['p75']

print(f"\n75th percentile threshold for total_poi_count in {state_filter}: {poi_percentile}")

# Filter to top 25% by total POI count
top_h3_cells = h3_features.filter(
    (col("total_poi_count") >= poi_percentile) &
    (col("total_poi_count") > 0)  # Ensure we have actual POIs
)

top_count = top_h3_cells.count()
print(f"H3 cells in top 25%: {top_count:,} ({100*top_count/total_cells:.1f}%)")

## Calculate H3 Centroids

In [ ]:
# Calculate H3 cell centroids for lat/lon
h3_with_centroids = top_h3_cells.select(
    col("h3_cell_id"),
    col("state_abbr"),
    col("total_poi_count"),
    col("population"),
    col("urbanity"),
    col("urbanity_category"),
    expr("h3_centeraswkt(h3_cell_id)").alias("center_wkt")
).withColumn(
    "center_point", expr("ST_GeomFromWKT(center_wkt, 4326)")
).withColumn(
    "latitude", expr("ST_Y(center_point)")
).withColumn(
    "longitude", expr("ST_X(center_point)")
).drop("center_wkt", "center_point")

print(f"Calculated centroids for {h3_with_centroids.count():,} H3 cells")
display(h3_with_centroids.limit(5))

## Load Current Stores for Distance Calculation

In [ ]:
# Load current stores from ALL states for distance calculation
# This ensures we calculate distance to the nearest store even if it's in a neighboring state
current_stores = spark.table(current_stores_table).select(
    col("location_id").alias("store_id"),
    col("state").alias("store_state"),
    col("latitude").alias("store_lat"),
    col("longitude").alias("store_lon")
)

store_count = current_stores.count()
print(f"Loaded {store_count} current stores (all states) for distance calculation")

# Show store distribution by state
print("\nCurrent stores by state:")
display(current_stores.groupBy("store_state").count().orderBy("store_state"))

if store_count == 0:
    print(f"\n⚠️  WARNING: No current stores found")
    print(f"Distance to nearest store will be set to NULL")

## Calculate Haversine Distance to Nearest Current Store

In [ ]:
# Haversine distance calculation using Spark SQL
# Formula: 2 * R * arcsin(sqrt(sin²((lat2-lat1)/2) + cos(lat1)*cos(lat2)*sin²((lon2-lon1)/2)))
# R = 6371 km (Earth's radius)

if store_count > 0:
    # Cross join candidates with stores and calculate distances
    candidates_with_distances = h3_with_centroids.crossJoin(
        broadcast(current_stores)
    ).withColumn(
        "distance_km",
        expr("""
            2 * 6371 * asin(
                sqrt(
                    pow(sin(radians(store_lat - latitude) / 2), 2) +
                    cos(radians(latitude)) * cos(radians(store_lat)) *
                    pow(sin(radians(store_lon - longitude) / 2), 2)
                )
            )
        """)
    )
    
    # Get minimum distance per H3 cell
    min_distance_per_cell = candidates_with_distances.groupBy(
        "h3_cell_id", "state_abbr", "total_poi_count", "population", "urbanity", 
        "urbanity_category", "latitude", "longitude"
    ).agg(
        F.min("distance_km").alias("distance_to_nearest_current_store")
    )
    
    print(f"Calculated distances for {min_distance_per_cell.count():,} candidate locations")
    
    # Show distance distribution
    print("\nDistance to nearest current store (km):")
    display(min_distance_per_cell.select("distance_to_nearest_current_store").summary())
else:
    # No stores - set distance to NULL
    min_distance_per_cell = h3_with_centroids.withColumn(
        "distance_to_nearest_current_store", lit(None).cast("double")
    )
    print("No current stores found - distance set to NULL")

## Format Output Schema

In [ ]:
# Add row numbers for location_id
window_spec = Window.orderBy(F.desc("total_poi_count"))

whitespace_locations = min_distance_per_cell.withColumn(
    "row_num", row_number().over(window_spec)
).withColumn(
    "location_id", (lit(location_id_start) + col("row_num") - 1).cast("int")
).drop("row_num")

# Add standardized columns - use state_abbr from H3 features data
whitespace_locations = whitespace_locations.select(
    col("location_id"),
    lit("Expansion Candidate").alias("store_type"),
    col("latitude").cast("double"),
    col("longitude").cast("double"),
    lit("NA").alias("address"),
    lit("NA").alias("city"),
    lit("NA").alias("zip_code"),
    col("state_abbr").alias("state"),  # Use state from H3 features instead of hardcoding
    lit("US").alias("country_code"),
    lit("H3_CENTROID").alias("geo_accuracy"),
    F.round(col("distance_to_nearest_current_store"), 2).alias("distance_to_nearest_current_store"),
    col("h3_cell_id"),
    col("total_poi_count"),
    col("population"),
    col("urbanity")
)

# Add processing timestamp
whitespace_locations = whitespace_locations.withColumn(
    "processing_timestamp", F.current_timestamp()
)

print(f"Generated {whitespace_locations.count():,} whitespace locations")
print(f"\nOutput schema:")
whitespace_locations.printSchema()

## Write to Silver Table

In [ ]:
# Write to Delta table
(
    whitespace_locations
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written {whitespace_locations.count():,} whitespace locations to {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("WHITESPACE LOCATIONS VALIDATION")
print("=" * 80)

# Summary statistics
print("\nSummary Statistics:")
display(spark.sql(f"""
    SELECT
        COUNT(*) as total_locations,
        COUNT(DISTINCT h3_cell_id) as unique_h3_cells,
        MIN(location_id) as min_location_id,
        MAX(location_id) as max_location_id,
        ROUND(AVG(total_poi_count), 0) as avg_poi_count,
        ROUND(AVG(population), 0) as avg_population,
        ROUND(AVG(distance_to_nearest_current_store), 2) as avg_distance_to_store_km,
        ROUND(MIN(distance_to_nearest_current_store), 2) as min_distance_km,
        ROUND(MAX(distance_to_nearest_current_store), 2) as max_distance_km
    FROM {output_table}
"""))

# Distribution by urbanity
print("\nBy Urbanity:")
display(spark.sql(f"""
    SELECT
        urbanity,
        COUNT(*) as location_count,
        ROUND(AVG(total_poi_count), 0) as avg_poi_count,
        ROUND(AVG(population), 0) as avg_population,
        ROUND(AVG(distance_to_nearest_current_store), 2) as avg_distance_km
    FROM {output_table}
    GROUP BY urbanity
    ORDER BY location_count DESC
"""))

# Sample records
print("\nSample whitespace locations:")
display(spark.table(output_table).select(
    "location_id", "store_type", "latitude", "longitude", "state",
    "geo_accuracy", "distance_to_nearest_current_store", "total_poi_count",
    "population", "urbanity"
).orderBy("location_id").limit(10))

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)